# Getting Started with OTTER Bioinformatics Workflows on Google Colab

**What is OTTER?**  
OTTER (**O**rchestrated **T**ranscriptomic, **T**umor-xenograft (PDX/CDX), and **E**pigenomic **R**eporting Workflow) is a reproducible bioinformatics workflow platform for high-throughput sequencing data — including RRBS/WGBS DNA methylation sequencing, RNA-seq transcriptomics, and dual-species PDX/CDX tumor-xenograft models. It automates transforming raw sequencing reads (FASTQ) into quality reports, expression matrices, and single-base methylation tables.

In this interactive Google Colab walkthrough, you do not need to configure a local high-performance computing (HPC) cluster. In this walkthrough, you will inspect OTTER's modern architecture and four core capabilities end to end:

1. **Zero-Friction Toolchain Installation** — Download the official `otter-install` binary and run a real installation: deploying all static CLI binaries (workflow compiler, QC tools, alignment operators) and creating managed Conda runtime environments via `enva`.
2. **Understand & Simulate the Reference Registry** — Learn why modern bioinformatics workflows decouple reference genomes into immutable, checksum-verified registries. We will build an exact mock reference release in seconds.
3. **Automate Project Creation (`otter build`)** — Use real downsampled sequencing data (FASTQ fixtures) checked into the repository to scan sample pairs, verify and lock reference identities, and freeze an immutable run snapshot in one single command.
4. **Full-Pipeline DAG Verification (`--dry-run`)** — Perform a rehearsal across **every phase published by each of the four workflow families**, watching the Craftmake engine compile the complete directed acyclic graph (DAG) without consuming hours of compute.

> 💡 **Beginner Note (What this notebook does NOT do):**  
> - The reference genome here is a lightweight structural fixture: directory layout, checksum manifests, and permission seals follow the production contract, but the sequence payloads are placeholders (avoiding downloads that can reach tens of gigabytes).  
> - `--dry-run` compiles and verifies the task planner; it does not execute hours-long read alignment algorithms.  
> - Runtime requirement: **Free CPU only**. A GPU is not needed for this walkthrough.  
> - The repository stores this notebook without execution outputs; run it in Colab to verify the selected release and current external services.

## 0. Preflight & Environment Check

Google Colab allocates a fresh Linux virtual machine for your session. Before proceeding, we verify that the runtime environment is Linux x86_64 and that essential system utilities (`git`, `python3`, `curl`, `tar`) are present.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
import time


def sh(command, check=True, capture=False, stream=False, echo=None, timeout=None):
    """Run a shell command, echoing it so the notebook reads as a transcript.

    capture=True collects output and prints it when the command finishes, which suits a
    short command whose output is a block to read. stream=True echoes output live, which
    is what a multi-minute step needs: a captured long command shows nothing and reads
    as a hang. echo=False captures without printing, for a command whose output the
    caller parses and summarises instead — a Craftmake plan envelope is hundreds of
    kilobytes of JSON, and printing it buries everything around it.
    """
    if echo is not None:
        capture = capture or not stream
    print(f"$ {command}")
    if stream:
        started = time.monotonic()
        process = subprocess.Popen(
            command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        for line in process.stdout:
            print(line.rstrip())
        returncode = process.wait(timeout=timeout)
        print(f"  (exit {returncode} after {time.monotonic() - started:.0f}s)")
        if check and returncode != 0:
            raise subprocess.CalledProcessError(returncode, command)
        return subprocess.CompletedProcess(command, returncode)

    result = subprocess.run(
        command,
        shell=True,
        check=check,
        text=True,
        capture_output=capture or echo is False,
        timeout=timeout,
    )
    if capture and echo is not False and result.stdout.strip():
        print(result.stdout.rstrip())
    return result


print("python :", sys.version.split()[0])
print("kernel :", platform.system(), platform.release(), platform.machine())
for tool in ("git", "curl", "tar"):
    print(f"{tool:7}:", shutil.which(tool) or "MISSING")

assert platform.system() == "Linux", "The OTTER release binaries are Linux builds."
assert platform.machine() in ("x86_64", "amd64"), (
    f"This notebook uses the amd64 release assets, not {platform.machine()}."
)
print("\nPreflight OK.")

### Global Workspace & Scenario Configuration

All paths, analysis scenarios, and parameters are organized centrally in this cell:
- **Four Core Scenarios**: RRBS (reduced representation bisulfite sequencing), RNA-seq (gene expression), BS-PDX (bisulfite sequencing of patient-derived xenografts), and RNA-PDX (RNA-seq of patient-derived xenografts).
- **Sequencing Fixtures**: Authentic downsampled FASTQ libraries (SRR accessions) stored directly in the repository.
- **Working Directory**: `/content/otter-colab` on Colab's fast local scratch volume.

In [ ]:
from pathlib import Path

# The public repository that hosts the releases and the test fixtures.
REPO = "otterlab-bio/otter"
RELEASE_TAG = "latest"          # pin, e.g. "v1.2.0", for a reproducible run

# Where everything is staged. /content is Colab's persistent-for-the-session volume;
# /tmp is RAM-backed and would be lost between cells.
WORK = Path("/content/otter-colab")
INSTALL_DIR = WORK / "bin"       # holds the released binaries
REPO_DIR = WORK / "repo"         # a shallow clone, for fixtures and the e2e script
REGISTRY = WORK / "registry"     # the simulated reference registry
PROJECTS = WORK / "projects"     # one authored project per scenario

for directory in (WORK, INSTALL_DIR, REGISTRY, PROJECTS):
    directory.mkdir(parents=True, exist_ok=True)

print("repository :", REPO)
print("release    :", RELEASE_TAG)
print("work dir   :", WORK)

# The fixtures the scenarios are authored from. Each is a real downsampled library
# checked into the repository, paired with the reference role that satisfies it.
#
# The release labels are the ones the published dataset actually carries, so every
# selection here can be replaced by a real fetch without changing the identifier —
# see "Downloading a real reference" below.
SCENARIOS = {
    "rrbs": {
        "mode": "RRBS",
        "accession": "SRR31480456",
        "references": {"primary": "hg19@GRCh37.p13-gencode-v19"},
    },
    "rnaseq": {
        "mode": "RNASEQ",
        "accession": "SRR018258",
        "references": {"primary": "hg38@GRCh38-gencode-v44"},
    },
    "bs-pdx": {
        "mode": "RRBS",
        "accession": "SRR36187610",
        "references": {
            "graft": "hg38@GRCh38-gencode-v44",
            "host": "mm10@GRCm38-gencode-M25",
        },
    },
    "rna-pdx": {
        "mode": "RNASEQ",
        "accession": "SRR30880970",
        "references": {
            "graft": "hg38@GRCh38-gencode-v44",
            "host": "mm10@GRCm38-gencode-M25",
        },
    },
}

print("\nscenarios:", ", ".join(SCENARIOS))

## 1. Real Installation via `otter-install`

Bioinformatics software management is notoriously prone to dependency conflicts and system library mismatches.  
OTTER solves this with a standalone, statically compiled Go installer (`otter-install`):
1. Downloads pre-compiled static binaries for all pipeline tools (independent of host OS dynamic libraries);
2. Deploys the built-in Craftmake workflow catalog;
3. Drives `enva` (a rattler/conda environment manager) to solve and create isolated runtime environments (`otter-core` and `otter-snakemake`).

Key flags used:
- `-install-dir`: Keeps the entire installation contained within Colab's session workspace.
- `-non-interactive`: Accepts all defaults automatically for clean, hands-off execution.

In [ ]:
installer_url = (
    f"https://github.com/{REPO}/releases/{RELEASE_TAG}/download/"
    "otter-install-linux-amd64-static"
)
installer_path = WORK / "otter-install"

sh(f"curl -fsSL -o {installer_path} {installer_url}")
installer_path.chmod(0o755)
print(f"\ninstaller: {installer_path} ({installer_path.stat().st_size:,} bytes)")

### Step 1: Preview the Installation Plan (`-dry-run`)

Reliable infrastructure tools allow users to inspect actions before writing to disk.  
The `-dry-run` flag prints every file download, extraction path, and environment setup step without modifying the system.

In [ ]:
# Assign to a name rather than leaving the call as the cell's last expression. A bare
# call in a Jupyter cell is auto-displayed as its repr, which re-prints the whole
# captured stdout as one escaped line and buries the readable output above it.
plan = sh(
    f"{installer_path} -dry-run -non-interactive "
    f"-releases-repo {REPO} -install-dir {INSTALL_DIR}",
    capture=True,
)

### Step 2: Execute Real Installation

Now let's perform the real installation!
- **Environment Creation Time**: When creating full Conda environments (default `SKIP_ENVS = False`), `conda`/`enva` will download real bioinformatics packages (FastQC, Bismark, Bowtie2, STAR, etc.), which typically takes 3–5 minutes. Live logs will stream below.
- ⚡ **Quick Preview Tip**: If you are short on time and only want to explore OTTER's project authoring, snapshot resolution, and planning capabilities without waiting for Conda packages, set `SKIP_ENVS = True` below (installs all binary tools in ~15 seconds).

In [ ]:
SKIP_ENVS = False  # set True to skip the multi-minute environment creation

skip_flag = "-skip-envs" if SKIP_ENVS else ""

# stream=True, not capture=True: this step downloads a full bioinformatics stack and
# can take several minutes. Capturing it would show nothing until it finished and read
# as a hang. Assigned rather than left bare so Jupyter does not also display its repr.
install = sh(
    f"{installer_path} -non-interactive {skip_flag} "
    f"-releases-repo {REPO} -install-dir {INSTALL_DIR}",
    stream=True,
)

### Step 3: Verify Installed Binaries & Tool Versions

We add the installation directory to `$PATH` and verify that all tools (`otter`, `craftmake`, `enva`, `fastqcx`, etc.) report their versions properly.

In [ ]:
os.environ["PATH"] = f"{INSTALL_DIR}:{os.environ['PATH']}"

print("=== installed binaries ===")
for binary in sorted(INSTALL_DIR.iterdir()):
    if binary.is_file() and os.access(binary, os.X_OK):
        print(f"  {binary.name}")

print("\n=== otter and craftmake versions ===")
sh("otter --version", capture=True)
sh("craftmake --version", capture=True)

print("=== workflow catalog deployed by the installer ===")
catalog = INSTALL_DIR / "workflows"
print(f"  {catalog}: {sorted(p.name for p in catalog.iterdir()) if catalog.is_dir() else 'MISSING'}")

In [ ]:
print("=== enva environments ===")
if SKIP_ENVS:
    print("  skipped (-skip-envs)")
else:
    sh("enva list", capture=True, check=False)

    # Prove otter-core is usable rather than merely listed: this is the difference
    # between "an environment directory exists" and "the environment works".
    print("\n=== does otter-core actually run a tool? ===")
    result = sh("enva run otter-core -- fastqc --version", capture=True, check=False)
    print("  reachable:", result.returncode == 0)

## 2. Retrieve Sequencing Fixtures & Registry Builder

Next, we obtain the actual sequencing test data needed for pipeline authoring. We perform a shallow git clone (fetching only the latest commit, which takes just a few seconds):
- **Real Downsampled Data**: Lightweight FASTQ files located under `testdata/` (preserving authentic sequencing reads and Phred quality scores, but truncated to a few thousand lines for rapid testing).
- **`stub-registry` Utility**: A small helper program that generates an offline reference registry using the production schema and validation rules. The cell below automatically configures Go if it is missing.

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    # Shallow and without submodules: this clone is only for the fixtures and the
    # rehearsal script, and the toolchain comes from the release.
    sh(
        f"git clone --depth 1 --no-recurse-submodules "
        f"https://github.com/{REPO}.git {REPO_DIR}",
        stream=True,
    )
else:
    print(f"reusing clone at {REPO_DIR}")

print()
sh(f"git -C {REPO_DIR} log --oneline -1", capture=True)

fixture_root = REPO_DIR / "testdata/gate6/craftmake-downsample-20260906/fastq"
print(f"\nfixtures: {fixture_root}")
print("  accessions:", sorted(p.name for p in fixture_root.iterdir()))

In [ ]:
# stub-registry is the only program this notebook compiles.
if shutil.which("go") is None:
    print("Go not found; installing it (a few seconds).")
    sh("apt-get -qq update && apt-get -qq install -y golang-go", check=False)

go_path = shutil.which("go")
print("go:", go_path or "STILL MISSING")
if go_path:
    # Assigned rather than left bare so Jupyter does not also display its repr.
    go_version = sh("go version", capture=True)

In [ ]:
stub_registry = INSTALL_DIR / "stub-registry"

if go_path is None:
    raise RuntimeError(
        "stub-registry needs a Go toolchain. Install Go, or pre-build the binary and place "
        f"it at {stub_registry}."
    )

sh(f"cd {REPO_DIR} && go build -o {stub_registry} ./internal/e2esupport/cmd/stub-registry")
print(f"\nstub-registry: {stub_registry.stat().st_size:,} bytes")

## 3. Understand & Simulate the Reference Registry

### What are Reference Genomes and Algorithm Indexes?
Before high-throughput sequencing reads can be quantified, they must be aligned against a standard "dictionary" — the reference genome (e.g., human hg38/hg19, mouse mm10). To align millions of reads in seconds, alignment tools (STAR, Bowtie2, Bismark) must pre-compute massive suffix arrays and Burrows-Wheeler transform indexes.

### Why does OTTER use an Immutable Reference Registry?
In traditional workflows, genome files and indexes are often scattered across various ad-hoc directories, leading to lost versions, broken scripts, and unreproducible science.  
OTTER introduces an enterprise-grade **centralized Reference Registry** architecture:
1. **Global Immutability**: Each reference release is sealed read-only (`chmod 0444/0555`);
2. **Cryptographic Manifests**: Every single file is indexed in `manifest.json` and `checksums.sha256` with its unique SHA-256 hash digest;
3. **Decoupled Architecture**: Analysis projects record only the logical reference identifier and manifest hash (e.g., `hg38@GRCh38-gencode-v44`). Reference data stays in the registry and is never duplicated inside user project directories.

### Why does this Colab use a Simulated Reference?
- **Disk Constraints**: Real mammalian reference indexes are massive (e.g., an uncompressed human hg38 STAR index alone exceeds 30 GB; a full reference suite can occupy tens of gigabytes). Free Colab instances only provide ~100 GB of temporary disk space, which would be exhausted by downloading real reference suites.
- **Validating Architecture**: Using `stub-registry`, we generate a simulated reference release in less than a second that features **the same registry schema, permission model, and checksum-manifest contract**. This exercises OTTER's verification and planning logic without claiming scientific equivalence to a real reference.

---

### 💡 How to Download Real Reference Genomes in Production
When working on your local workstation, lab server, or HPC cluster with sufficient storage, OTTER provides three production methods to acquire real reference genomes:

#### Option A: Automated Download of Pre-Built Indexes (Recommended)
The OTTER project hosts pre-built reference packages on Hugging Face, saving you hours of compute-heavy index compilation:
```text
https://huggingface.co/datasets/fallingstar10/xdxtools-genomes
Available pre-built releases:
  - hg19@GRCh37.p13-gencode-v19  (Human hg19 / GRCh37)
  - hg38@GRCh38-gencode-v44      (Human hg38 / GRCh38)
  - mm10@GRCm38-gencode-M25      (Mouse mm10 / GRCm38)
  - mm39@GRCm39-gencode-vM39     (Mouse mm39 / GRCm39)
  - mm9@NCBIM37-gencode-M1       (Mouse mm9 / NCBIM37)
```
In your production environment, set the environment variables and let `otter-install` download and extract them:
```bash
# 1. Specify which releases to fetch (comma-separated for multiple)
export OTTER_REFERENCE_FETCH_RELEASES="hg38@GRCh38-gencode-v44"

# 2. Specify the target registry root directory on your server
export OTTER_REFERENCE_FETCH_REGISTRY_ROOT=/shared/otter/references

# 3. Run automated download (fetches archives, validates checksums, unpacks)
otter-install -reference-fetch -non-interactive
```
*💡 Pro-tip: If disk space is limited, you can exclude large indexes (such as 30 GB STAR indexes) by setting `OTTER_REFERENCE_FETCH_ASSETS="bismark,bowtie2,fasta,annotations"`.*

#### Option B: Build from Source (For Novel or Custom Genomes)
For custom species or self-assembled genomes, run `otter reference build` with raw FASTA and GTF files. The toolchain will automatically invoke `samtools`, `bismark_genome_preparation`, `bowtie2-build`, and `STAR` to compile all indexes and publish an immutable release:
```bash
otter reference build   --id my_species   --release custom-v1   --organism "Danio rerio"   --assembly "GRCz11"   --fasta /path/to/genome.fa.gz   --gtf /path/to/genes.gtf.gz
```

#### Option C: Direct Web Download
Visit `https://huggingface.co/datasets/fallingstar10/xdxtools-genomes`, browse the file tree, and download the `.tar.gz` archives manually into `<registry-root>/genomes/<id>/<release>/`.

---

### Seamless Usage in Projects
Regardless of which method you used to acquire the reference genome, referencing it in your projects is identical — simply pass `--reference-primary <id>@<release>` to `otter build` or `otter create`, and OTTER will automatically mount the verified indexes!

In [ ]:
# Every reference the scenarios select. Both species are needed because the PDX scenarios
# declare a graft and a host.
#
# These must match the selections in SCENARIOS above: a scenario can only select a release
# that exists in the registry, so the two lists are one contract. The labels are the ones
# the published dataset carries, which is what lets a reader swap the simulated registry
# for a real fetch without editing the selections.
REFERENCES = [
    {
        "id": "hg19",
        "release": "GRCh37.p13-gencode-v19",
        "organism": "Homo sapiens",
        "assembly": "GRCh37.p13",
        "aliases": "hg19,human,grch37",
    },
    {
        "id": "hg38",
        "release": "GRCh38-gencode-v44",
        "organism": "Homo sapiens",
        "assembly": "GRCh38",
        "aliases": "hg38,human,grch38",
    },
    {
        "id": "mm10",
        "release": "GRCm38-gencode-M25",
        "organism": "Mus musculus",
        "assembly": "GRCm38",
        "aliases": "mm10,mouse,grcm38",
    },
]

for reference in REFERENCES:
    sh(
        f"{stub_registry} --registry-root {REGISTRY} "
        f"--id {reference['id']} --release {reference['release']} "
        f"--organism '{reference['organism']}' --assembly {reference['assembly']} "
        f"--alias {reference['aliases']}"
    )
    print()

In [ ]:
import json

print("=== registry layout ===")
sh(f"find {REGISTRY} -maxdepth 4 -mindepth 3 | sort", capture=True)

release_dir = REGISTRY / "genomes/hg19/GRCh37.p13-gencode-v19"
print("=== reference.yaml (the release identity) ===")
print("\n".join(
    f"  {line}" for line in (release_dir / "reference.yaml").read_text().splitlines()[:14]
))

manifest = json.loads((release_dir / "manifest.json").read_text())
print(f"\n=== manifest.json: {len(manifest)} entries ===")
for entry in manifest[:5]:
    print(f"  {entry['path']}")
print(f"  ... {max(0, len(manifest) - 5)} more")

# The release contract is reference.yaml + manifest.json + checksums.sha256. Report
# which of them the fixture wrote rather than assuming all three: the stub and the
# production publisher are separate code paths, and a notebook that asserts one
# implementation's output goes down on the other's.
print("\n=== contract files written by the fixture ===")
contract_files = ["reference.yaml", "manifest.json", "checksums.sha256"]
for name in contract_files:
    path = release_dir / name
    if path.is_file():
        print(f"  {name:<20} {path.stat().st_size:>6} bytes")
    else:
        print(f"  {name:<20} MISSING")

checksums_path = release_dir / "checksums.sha256"
if checksums_path.is_file():
    print("\n=== checksums.sha256 covers the release identity ===")
    print("\n".join(
        f"  {line}"
        for line in checksums_path.read_text().splitlines()[:4]
    ))
    print("\n  Verified against every file it names:")
    # Assigned, not left bare: a bare call as the cell's last expression is
    # auto-displayed as its repr, which re-prints stdout as one escaped line.
    checksum_check = sh(f"cd {release_dir} && sha256sum -c checksums.sha256 | tail -3", capture=True)
else:
    print(
        "\nchecksums.sha256 is absent, so sha256sum -c cannot be demonstrated here."
        "\nThe file is part of the release contract, so its absence is a fixture gap:"
        "\nreport it rather than treating the release as complete."
    )

## 4. Single-Command Project Creation via `otter build`

In traditional bioinformatics pipelines, setting up a new analysis project typically requires:
1. Creating directories manually and copying rule scripts;
2. Writing lengthy YAML configuration files prone to indentation mistakes;
3. Manually inspecting FASTQ filenames and drafting sample manifests;
4. Verifying reference paths by hand.

The new **`otter build`** command combines all of these steps into **a single automated command**:
- **Automatic Sample Pairing**: Discovers `_R1` and `_R2` FASTQ files, matches sample IDs against `pdata.csv`, and derives adapter sequences;
- **Cryptographic Reference Verification**: Checks the declared reference against the registry and locks its SHA-256 digest into `references.lock.yaml`;
- **Standardized Project Scaffolding**: Writes `project.yaml` and `samples.tsv`;
- **Immutable Run Snapshot Generation**: Freezes a timestamped, read-only `run.yaml` snapshot containing all parameters and paths.

In the cell below, we execute `otter build` for all four distinct scenarios (RRBS, RNA-seq, BS-PDX, RNA-PDX)!

In [ ]:
def stage_inputs(scenario, accession):
    """Copy one fixture pair under the names `otter create` expects, plus its pdata."""
    project_dir = PROJECTS / scenario
    fastq_dir = project_dir / "fastq"
    fastq_dir.mkdir(parents=True, exist_ok=True)

    source = fixture_root / accession
    shutil.copyfile(source / "R1.fastq.gz", fastq_dir / f"{accession}_R1.fastq.gz")
    shutil.copyfile(source / "R2.fastq.gz", fastq_dir / f"{accession}_R2.fastq.gz")

    pdata = project_dir / "pdata.csv"
    pdata.write_text(
        "sampleid,inline_barcode_sequence,condition\n" f"{accession},,case\n"
    )
    return project_dir, fastq_dir, pdata


def reference_arguments(references):
    """Render the role flags. A primary is one role; PDX is graft plus host."""
    if "primary" in references:
        return f"--reference-primary {references['primary']}"
    return (
        f"--reference-graft {references['graft']} "
        f"--reference-host {references['host']}"
    )


snapshots = {}

for scenario, spec in SCENARIOS.items():
    print("=" * 72)
    print(f"scenario: {scenario}  (mode={spec['mode']}, fixture={spec['accession']})")
    print("=" * 72)

    project_dir, fastq_dir, pdata = stage_inputs(scenario, spec["accession"])
    references = reference_arguments(spec["references"])

    result = sh(
        f"otter build --project-root {project_dir} "
        f"--fastq {fastq_dir} --pdata {pdata} --mode {spec['mode']} "
        f"--jobid {scenario} "
        f"--reference-root {REGISTRY} {references} --backend local",
        capture=True,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(f"otter build failed for {scenario} (exit {result.returncode})")

    # `build` prints the snapshot path on its own line, last.
    snapshot_path = Path(result.stdout.strip().splitlines()[-1].strip())
    assert snapshot_path.name == "run.yaml", f"unexpected snapshot path: {snapshot_path}"
    assert snapshot_path.exists(), f"reported snapshot does not exist: {snapshot_path}"
    snapshots[scenario] = snapshot_path

    for artifact in ("project.yaml", "samples.tsv", "references.lock.yaml"):
        assert (project_dir / artifact).exists(), f"{scenario} is missing {artifact}"

    print(f"  snapshot: {snapshot_path.relative_to(WORK)}")
    print()

print(f"authored {len(snapshots)} scenarios: {', '.join(snapshots)}")

### Inspect the Generated Project Structure

Let's examine the generated structure of the `rrbs` project:
1. **`project.yaml`**: High-level analysis intent (scenario: RRBS, executor: Craftmake);
2. **`samples.tsv`**: Sample manifest with project-relative paths to FASTQ files;
3. **`references.lock.yaml`**: Cryptographic lock of the reference release and manifest digest;
4. **`workflows/rules/`**: Snakemake rule assets pinned cleanly inside the project directory, keeping the project root tidy.

In [ ]:
sample_project = PROJECTS / "rrbs"

print("=== project layout ===")
sh(f"find {sample_project} -maxdepth 2 -type d | sort", capture=True)

print("=== project.yaml ===")
print("\n".join(
    f"  {line}" for line in (sample_project / "project.yaml").read_text().splitlines()
))

print("\n=== samples.tsv (paths are project-relative) ===")
print("\n".join(
    f"  {line}" for line in (sample_project / "samples.tsv").read_text().splitlines()
))

print("\n=== references.lock.yaml (the locked digest) ===")
print("\n".join(
    f"  {line}"
    for line in (sample_project / "references.lock.yaml").read_text().splitlines()
))

print("\n=== the rules are pinned under workflows/, not at the root ===")
print("  workflows/rules exists:", (sample_project / "workflows/rules").is_dir())
print("  project-root rules/ exists:", (sample_project / "rules").exists())

## 5. Dry-Run Verification: Planning Every Workflow Phase

### Why `--dry-run` is Essential
Real high-throughput sequencing workflows consume hours of CPU time. Discovering a syntax error or a missing intermediate input 5 hours into a run is frustrating and costly.

OTTER's `--dry-run` provides **pre-execution planning and contract validation**:
- Reads the immutable `run.yaml` snapshot;
- Directs Craftmake to resolve the directed acyclic graph (DAG) across **all phases**:
  - `step1`: Quality trimming and adapter removal (QC & Trimming)
  - `step2`: Sequence alignment (Alignment)
  - `step2-check`: Alignment artifact validation
  - `step3`: Methylation extraction or gene count matrix generation (Quantification)
  - `step3-check`: Quantification outcome checks
  - `publish`: Final artifact verification and structured publication
- Validates all inter-task dependency relationships.

**No heavy alignment algorithms are executed, and no raw data is modified.** A successful plan confirms that the snapshot and declared DAG compile; it does not validate scientific outputs or runtime tool behavior.

In [ ]:
import json

# Named explicitly rather than left to PATH: the resolver falls back to a PATH
# lookup, and a notebook that reorders its environment should not fail on that.
craftmake_binary = INSTALL_DIR / "craftmake"
catalog = INSTALL_DIR / "workflows"


def phases_for(workflow):
    """List a workflow's published phases, in order, from the deployed catalog.

    Read from the catalog rather than hardcoded: the phase set differs per workflow
    (RNA-seq has no step3), and a hardcoded list would silently stop covering a new
    phase the moment one is added.
    """
    order = {"step1": 0, "step2": 1, "step2-check": 2, "step3": 3, "step3-check": 4, "publish": 5}
    names = [p.stem for p in (catalog / workflow).glob("*.yaml")]
    return sorted(names, key=lambda name: (order.get(name, 99), name))


def plan_phase(scenario, snapshot_path, phase):
    """Plan one phase and return its envelope.

    echo=False, because a plan envelope is hundreds of kilobytes of JSON: printing it
    would bury the summary this cell exists to produce. The task count is the useful
    part, and it is printed below.
    """
    result = sh(
        f"otter run --config {snapshot_path} "
        f"--executor craftmake --phase {phase} "
        f"--dry-run --foreground "
        f"--craftmake-binary {craftmake_binary} --catalog {catalog}",
        echo=False,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"dry run failed for {scenario} {phase} (exit {result.returncode})")
    return json.loads(result.stdout)


# Every phase is planned, not just step1. A run resolves to a multi-phase workflow,
# so planning one phase proves one phase; the later phases consume the earlier ones'
# outputs, and that is where a resolution problem actually shows up.
plan_results = {}
total_tasks = 0

for scenario, snapshot_path in snapshots.items():
    print("=" * 72)
    print(f"planning: {scenario}")
    print("=" * 72)

    # Discover the workflow from the snapshot's own step1 plan, so the phase list
    # comes from what this run actually selects rather than from a naming convention.
    first_envelope = plan_phase(scenario, snapshot_path, "step1")
    workflow = first_envelope["data"]["workflow"]
    phase_names = phases_for(workflow)
    print(f"  workflow: {workflow}")
    print(f"  phases  : {', '.join(phase_names)}")
    print()

    scenario_plans = {}
    for phase in phase_names:
        envelope = (
            first_envelope
            if phase == "step1"
            else plan_phase(scenario, snapshot_path, phase)
        )

        assert envelope.get("command") == "plan", envelope.get("command")
        assert envelope.get("ok") is True, envelope
        tasks = envelope.get("data", {}).get("tasks", [])
        assert tasks, f"{scenario} {phase} returned no tasks"

        scenario_plans[phase] = len(tasks)
        total_tasks += len(tasks)
        print(f"    {phase:<12} {len(tasks):>3} tasks")
    print()

    plan_results[scenario] = {"workflow": workflow, "phases": scenario_plans}

print(f"planned every phase of {len(plan_results)} scenarios, {total_tasks} tasks in total")
print("no task was executed")

## 6. Execution Summary

Congratulations! You have completed the complete OTTER lifecycle: toolchain installation, reference registry simulation, single-command project building, and multi-phase dry-run planning.  
The summary table below displays the workflow family and total planned tasks for each scenario.

In [ ]:
from datetime import datetime, timezone

print("=" * 72)
print("OTTER Colab walkthrough -- summary")
print("=" * 72)
print(f"finished     : {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} UTC")
print(f"repository   : {REPO}")
print(f"release      : {RELEASE_TAG}")
print(f"environments : {'skipped' if SKIP_ENVS else 'otter-core (created)'}")
print()

header = f"{'scenario':<10} {'mode':<8} {'fixture':<12} {'workflow':<17} {'phases':>6} {'tasks':>6}"
print(header)
print("-" * len(header))
for scenario, spec in SCENARIOS.items():
    result = plan_results[scenario]
    phase_tasks = result["phases"]
    print(
        f"{scenario:<10} {spec['mode']:<8} {spec['accession']:<12} "
        f"{result['workflow']:<17} {len(phase_tasks):>6} {sum(phase_tasks.values()):>6}"
    )

print()
print("Every scenario resolved to an immutable snapshot, and every phase of each")
print("workflow reached Craftmake's planner. No bioinformatics tool was executed and")
print("no result is scientific.")
print()
print(f"artifacts under {WORK}:")
print(f"  {INSTALL_DIR}   released binaries")
print(f"  {REGISTRY}      simulated reference registry")
print(f"  {PROJECTS}      authored projects and resolved runs")

## Next Steps & Further Exploration

### 1. Run the Complete 101-Stage Offline Rehearsal Script
The repository includes an extensive rehearsal test suite (`scripts/e2e/otter_e2e.sh`) that validates cross-track guards, SLURM site profiling, executor pairing, and artifact-by-artifact equivalence between `otter build` and manual step-by-step authoring:
```bash
cd /content/otter-colab/repo
bash scripts/e2e/otter_e2e.sh   --otter            /content/otter-colab/bin/otter   --craftmake        /content/otter-colab/bin/craftmake   --stub-registry    /content/otter-colab/bin/stub-registry   --installer        /content/otter-colab/otter-install   --craftmake-catalog /content/otter-colab/bin/workflows
```

### 2. Execute Real Computing on Your Server
When running on a server or HPC cluster with real sequencing data, remove `--dry-run` to launch actual background execution:
```bash
# Launch background task execution
otter run --config /path/to/run.yaml --executor craftmake --phase step1 --backend local

# Monitor task status and stream live logs
otter task list
otter task logs <task-id> --follow
```

### 3. Read the Documentation
- [Official OTTER User Manual](https://github.com/otterlab-bio/otter/blob/main/docs/manual/README.md)
- [Reference Registry Migration Guide](https://github.com/otterlab-bio/otter/blob/main/docs/manual/08-reference-migration.md)

---
*Happy analyzing with OTTER!*